## renomear tabelas bronze e silver

In [0]:
database_name_old = "workspace.default"
database_name_new = "workspace.bronze"
table_name_old = "queimadas_raw"
table_name_new = "queimadas"

spark.sql(
    f"""ALTER TABLE {database_name_old}.{table_name_old}
    RENAME TO {database_name_new}.{table_name_new};
"""
)

In [0]:
database_name_old = "workspace.default"
database_name_new = "workspace.silver"
table_name_old = "queimadas_cleaned"
table_name_new = "queimadas"

spark.sql(
    f"""ALTER TABLE {database_name_old}.{table_name_old}
    RENAME TO {database_name_new}.{table_name_new};
"""
)

## fazer tabela gold

In [0]:
from pyspark.sql import functions as F

silver_database = 'workspace.silver'
silver_table = 'queimadas'

df_gold = (
    spark.table(f"{silver_database}.{silver_table}")
    .groupBy("sigla_uf", "bioma", "mes", "ano")
    .agg(
        F.count("*").alias("total_focos"),
        F.avg("latitude").alias("lat_media"),
        F.avg("longitude").alias("lon_media")
    )
)

display(df_gold.limit(100))


In [0]:
gold_database = "workspace.gold"
gold_table = "queimadas_uf"


(
    df_gold.write.format("delta") #pode ser parquet, json...
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable(f"{gold_database}.{gold_table}")
)

## Funcionalidades delta

In [0]:
%sql
-- histórico

describe history workspace.gold.queimadas_uf

In [0]:
%sql
-- time travel

-- -- Versão atual
-- SELECT * FROM workspace.gold.queimadas_uf VERSION AS OF 1 LIMIT 5;

-- -- 'Rollback' para versão anterior
SELECT * FROM workspace.gold.queimadas_uf VERSION AS OF 0 LIMIT 5;

In [0]:
%sql
-- governança

DESCRIBE DETAIL workspace.gold.queimadas_uf;